In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.proxy import Proxy, ProxyType
import time
import os

In [ ]:
KONU_ID = 81
DOWNLOAD_PATH = 'files_ydu'
FILE_EXT = "xls" # or "csv"
LOCALE = "tr"
HOMEPAGE = f"https://biruni.tuik.gov.tr/medas/?kn={KONU_ID}&locale={LOCALE}"

In [ ]:
prefs = {
   "download.default_directory": DOWNLOAD_PATH,
   "savefile.default_directory": DOWNLOAD_PATH
}
options = webdriver.ChromeOptions()
options.add_experimental_option('prefs', prefs)
prxy = Proxy()
prxy.proxy_type = ProxyType.MANUAL
prxy.http_proxy = "proxy.address:8080"
prxy.ssl_proxy = "proxy.address:8080"
options.proxy = prxy
driver = webdriver.Chrome(options=options)

Error sending stats to Plausible: error sending request for url (https://plausible.io/api/event)


In [4]:
if not os.path.exists(DOWNLOAD_PATH):
    os.makedirs(DOWNLOAD_PATH)

In [5]:
def check_loading():
    time.sleep(1)
    loading = driver.find_elements(By.CLASS_NAME, "z-loading-indicator")
    while len(loading) > 0:
        time.sleep(1)
        loading = driver.find_elements(By.CLASS_NAME, "z-loading-indicator")

In [6]:
def add_measurements():
    toolbar_btns = driver.find_elements(By.CLASS_NAME, "z-toolbarbutton")
    add_btn = [btn for btn in toolbar_btns if (LOCALE == 'tr' and btn.get_attribute("title") == "Göstergeleri Ekle") or (LOCALE == 'en' and btn.get_attribute("title") == 'Add Measurement(s)')][0]
    add_btn.click()

In [7]:
def download_file(itemtxt, i):    
    download_btns = driver.find_elements(By.TAG_NAME, "img")
    valid_dl_btn = [btn for btn in download_btns if btn.get_attribute("src") and btn.get_attribute("src").endswith(f"{FILE_EXT}.png")][0]
    valid_dl_btn.click()
    time.sleep(4)
    
    if not os.path.exists(f"{DOWNLOAD_PATH}"):
        os.makedirs(f"{DOWNLOAD_PATH}")
    new_file_name = f"{DOWNLOAD_PATH}\\{itemtxt.strip()} {i}.{FILE_EXT}"
    os.rename(f"{DOWNLOAD_PATH}\\pivot.{FILE_EXT}", new_file_name)

In [8]:
hierarchy = []
driver.get(HOMEPAGE)
check_loading()

litems = driver.find_elements(By.CLASS_NAME, "z-listitem")
for litem in litems:
    row = {'text': litem.text}

    litem.click()
    check_loading()

    warn_window = driver.find_elements(By.CLASS_NAME, "z-window")
    row["default"] = len(warn_window) > 0

    vboxes = driver.find_elements(By.CLASS_NAME, "z-vbox")
    dimensions = vboxes[0].find_elements(By.CLASS_NAME, "z-listitem")
    selected_dims = [d.text for d in dimensions if (LOCALE == 'tr' and d.get_attribute("title") == "Bu kırılımın seçilmesi zorunludur.") or (LOCALE == 'en' and d.get_attribute("title") == 'This dimension is required.')]
    selectable_dims = [d for d in dimensions if not ((LOCALE == 'tr' and d.get_attribute("title") == "Bu kırılımın seçilmesi zorunludur.") or (LOCALE == 'en' and d.get_attribute("title") == 'This dimension is required.'))]
    
    row['selected'] = selected_dims

    dims = []
    time.sleep(3)    
    if len(selectable_dims) == 1:
        row['selectable'] = [[selectable_dims[0].text]]
        hierarchy.append(row)
        continue
    
    for idx in range(len(selectable_dims)):
        if selectable_dims[idx].text in [g for d in dims for g in d]:
            continue
        selectable_dims[idx].click()
        time.sleep(2)
        dimensions = vboxes[0].find_elements(By.CLASS_NAME, "z-listitem")
        selectable_dims = [d for d in dimensions if not ((LOCALE == 'tr' and d.get_attribute("title") == "Bu kırılımın seçilmesi zorunludur.") or (LOCALE == 'en' and d.get_attribute("title") == 'This dimension is required.'))]
        grp = [sd.text for sd in selectable_dims if 'z-listitem-selected' in sd.get_attribute("class") or 'z-listitem-disabled' not in sd.get_attribute("class")]
        dims.append(grp)
        selectable_dims[idx].click()
        time.sleep(2)
        dimensions = vboxes[0].find_elements(By.CLASS_NAME, "z-listitem")
        selectable_dims = [d for d in dimensions if not ((LOCALE == 'tr' and d.get_attribute("title") == "Bu kırılımın seçilmesi zorunludur.") or (LOCALE == 'en' and d.get_attribute("title") == 'This dimension is required.'))]
    
    row['selectable'] = dims
    hierarchy.append(row)

        

In [10]:
for row in hierarchy:
    for i, dim in enumerate(row['selectable']):
        driver.get(HOMEPAGE)
        check_loading()
        
        litems = driver.find_elements(By.CLASS_NAME, "z-listitem")
        curr_item = [litem for litem in litems if litem.text == row['text']][0]
        curr_item.click()
        litem_text = curr_item.text
        check_loading()
        
        vboxes = driver.find_elements(By.CLASS_NAME, "z-vbox")
        dimensions = vboxes[0].find_elements(By.CLASS_NAME, "z-listitem")
        selected_dims = [d.text for d in dimensions if (LOCALE == 'tr' and d.get_attribute("title") == "Bu kırılımın seçilmesi zorunludur.") or (LOCALE == 'en' and d.get_attribute("title") == 'This dimension is required.')]
        selectable_dims = [d for d in dimensions if not ((LOCALE == 'tr' and d.get_attribute("title") == "Bu kırılımın seçilmesi zorunludur.") or (LOCALE == 'en' and d.get_attribute("title") == 'This dimension is required.'))]

        curr_dims = [d for d in selectable_dims if d.text in dim]
        for curr_dim_idx in range(len(curr_dims)):
            time.sleep(3)
            dimensions = vboxes[0].find_elements(By.CLASS_NAME, "z-listitem")
            selectable_dims = [d for d in dimensions if not ((LOCALE == 'tr' and d.get_attribute("title") == "Bu kırılımın seçilmesi zorunludur.") or (LOCALE == 'en' and d.get_attribute("title") == 'This dimension is required.'))]
            curr_dims = [d for d in selectable_dims if d.text in dim]
            selected_dims.append(curr_dims[curr_dim_idx].text)
            curr_dims[curr_dim_idx].click()
        
        time.sleep(2)
        if row['default']:
            add_measurements()
            check_loading()
        
        vbox_btns = vboxes[0].find_elements(By.TAG_NAME, "button")
        ok_btn = [btn for btn in vbox_btns if (LOCALE == 'tr' and btn.text == "Tamam") or (LOCALE == 'en' and btn.text == 'Ok')][0]
        ok_btn.click()
        check_loading()
        
        captions = driver.find_elements(By.CLASS_NAME, "z-caption-content")
        dim_boxes = [cap for cap in captions if cap.text.strip() in selected_dims]
        for dim_box in dim_boxes:
            list_opts = dim_box.find_element(By.XPATH, "../../..").find_elements(By.CLASS_NAME, "z-listitem")
            if (LOCALE == 'tr' and list_opts[0].text == '<Hepsi>') or (LOCALE == 'en' and list_opts[0].text == '<All>'):
                list_opts[0].click()
                time.sleep(1)
        
        add_measurements()
        check_loading()
        
        page_btns = driver.find_elements(By.TAG_NAME, "button")
        next_btn = [btn for btn in page_btns if (LOCALE == 'tr' and btn.text == "İleri") or (LOCALE == 'en' and btn.text == 'Forward')][0]
        next_btn.click()
        check_loading()
        
        time.sleep(2)
        rows_chbox = driver.find_elements(By.CLASS_NAME, "z-listheader-checkable")
        rows_chbox[0].click()
        check_loading()
        
        page2_btns = driver.find_elements(By.TAG_NAME, "button")
        next_btn2 = [btn for btn in page2_btns if (LOCALE == 'tr' and btn.text == "İleri") or (LOCALE == 'en' and btn.text == 'Forward')][0]
        next_btn2.click()
        check_loading()

        rows_chbox2 = driver.find_elements(By.CLASS_NAME, "z-listheader-checkable")
        rows_chbox2[1].click()
        check_loading()

        page3_btns = driver.find_elements(By.TAG_NAME, "button")
        report_btn = [btn for btn in page3_btns if (LOCALE == 'tr' and btn.text == "Rapor Oluştur") or (LOCALE == 'en' and btn.text == 'Create Report')][0]
        report_btn.click()
        check_loading()
        
        download_file(litem_text, i)
    
